# 04. 파일, 권한과 파이프라인


## Goal

파일을 안전하게 만들고 권한을 제한하며 파이프라인 실패를 감지합니다.


## Setup


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-04-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


## Steps

### 1. 제한된 권한으로 파일 생성


In [ ]:
%%bash
set -euo pipefail
umask 077
printf 'case_id=LAB-001\n' > "$BASH_LAB_DIR/evidence.txt"
ls -l "$BASH_LAB_DIR/evidence.txt"
test ! -x "$BASH_LAB_DIR/evidence.txt"


### 2. 표준 출력과 표준 오류 저장


In [ ]:
%%bash
set -euo pipefail
{
  printf 'collection started\n'
  printf 'sample warning\n' >&2
} >"$BASH_LAB_DIR/stdout.log" 2>"$BASH_LAB_DIR/stderr.log"
printf '%s\n' '--- stdout ---'
cat "$BASH_LAB_DIR/stdout.log"
printf '%s\n' '--- stderr ---'
cat "$BASH_LAB_DIR/stderr.log"


### 3. `pipefail`로 중간 실패 감지


In [ ]:
%%bash
set -uo pipefail
if bash -c 'printf data; exit 9' | wc -c > /dev/null; then
  printf '예상하지 못한 성공\n'
else
  printf '파이프라인 실패를 감지했습니다. status=%s\n' "$?"
fi


## Checks

- 민감한 실습 파일이 다른 사용자에게 쓰기 가능하지 않은가?
- 정상 출력과 오류 출력이 서로 다른 파일에 저장되었는가?
- 파이프의 첫 명령이 실패했을 때 전체 파이프라인도 실패했는가?


## Next Steps

행 단위 로그를 필터링하고 요약 보고서를 만듭니다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
